# 12 Beta Diversity and Body Mass Index (BMI)

We test whether fungal community composition is associated with body mass index (BMI) (`bmi_sample`). In Section 1, we treat BMI as a continuous predictor using its numeric values. In Section 2, we treat BMI as a categorical predictor and run overall and pairwise PERMANOVAs for the four BMI groups.

In each section, we compare whole-ASV and 16-mer Bray–Curtis distances using the same samples for both representations and both models. The adjusted models account for age, sex, and reported diet because all three may be associated with both BMI and fungal community composition.


## 1. BMI as a continuous predictor (numeric values)

## 1.1 Setup

### 1.1.1 Imports

In [16]:
import subprocess
from pathlib import Path

import pandas as pd
import qiime2 as q2
from skbio import DistanceMatrix
from qiime2 import Visualization, Artifact, Metadata

### 1.1.2 Notebook settings and paths

We define the input paths, the shared complete-case files, and the outputs for each feature representation and model.

In [17]:
# Shared Setup for continuous and categorical BMI analysis

## Shared inputs and output directories
metadata_path = "../data/raw/Final_metadata_with_GHSL_exclude_other_sex_with_BMI_groups.tsv"
kmer16_distance_path = "../data/processed_ignored/07_beta_diversity/kmer16_bray_curtis_distance.qza"
asv_distance_path = "../data/processed_ignored/07_beta_diversity/asv_bray_curtis_distance.qza"

out_dir = "../data/processed_ignored/12_beta_diversity_bmi"
vis_dir = "../data/processed/12_beta_diversity_bmi"


## Complete-case inputs for both models
adjusted_bmi_out_dir = f"{out_dir}/adjusted_bmi"

asv_bmi_distance_path = f"{adjusted_bmi_out_dir}/asv_bray_curtis_complete.qza"
kmer16_bmi_distance_path = f"{adjusted_bmi_out_dir}/kmer16_bray_curtis_complete.qza"

bmi_metadata_path = f"{adjusted_bmi_out_dir}/metadata_complete.tsv"


## Create directories
!mkdir -p "$out_dir" "$vis_dir" "$adjusted_bmi_out_dir"
!mkdir -p "$con_dir" "$cat_dir"


mkdir: : No such file or directory
mkdir: : No such file or directory


In [18]:

# Continuous BMI

con_dir = f"{vis_dir}/continuous_bmi"

## Overall PERMANOVA outputs
unadjusted_con_permanova_dir = f"{con_dir}/unadjusted_con_permanova"
adjusted_con_permanova_dir = f"{con_dir}/adjusted_con_permanova"

asv_con_unadjusted_overall_path = f"{unadjusted_con_permanova_dir}/asv_overall.qzv"
kmer16_con_unadjusted_overall_path = f"{unadjusted_con_permanova_dir}/kmer16_overall.qzv"

asv_con_adjusted_overall_path = f"{adjusted_con_permanova_dir}/asv_overall.qzv"
kmer16_con_adjusted_overall_path = f"{adjusted_con_permanova_dir}/kmer16_overall.qzv"


## Create directories
!mkdir -p "$con_dir"
!mkdir -p "$unadjusted_con_permanova_dir" "$adjusted_con_permanova_dir"

In [19]:
# Categorical BMI

cat_dir = f"{vis_dir}/categorical_bmi"

## Overall PERMANOVA outputs
unadjusted_cat_permanova_dir = f"{cat_dir}/unadjusted_cat_permanova"
adjusted_cat_permanova_dir = f"{cat_dir}/adjusted_cat_permanova"

asv_cat_unadjusted_overall_path = f"{unadjusted_cat_permanova_dir}/asv_overall.qzv"
kmer16_cat_unadjusted_overall_path = f"{unadjusted_cat_permanova_dir}/kmer16_overall.qzv"

asv_cat_adjusted_overall_path = f"{adjusted_cat_permanova_dir}/asv_overall.qzv"
kmer16_cat_adjusted_overall_path = f"{adjusted_cat_permanova_dir}/kmer16_overall.qzv"


## Pairwise PERMANOVA outputs
pairwise_input_dir = f"{adjusted_bmi_out_dir}/pairwise_inputs"

asv_cat_unadjusted_pairwise_path = f"{unadjusted_cat_permanova_dir}/asv_pairwise"
kmer16_cat_unadjusted_pairwise_path = f"{unadjusted_cat_permanova_dir}/kmer16_pairwise"

asv_cat_adjusted_pairwise_path = f"{adjusted_cat_permanova_dir}/asv_pairwise"
kmer16_cat_adjusted_pairwise_path = f"{adjusted_cat_permanova_dir}/kmer16_pairwise"


## Create directories
!mkdir -p "$cat_dir"
!mkdir -p "$unadjusted_cat_permanova_dir" "$adjusted_cat_permanova_dir"
!mkdir -p "$pairwise_input_dir" "$asv_cat_unadjusted_pairwise_path" "$kmer16_cat_unadjusted_pairwise_path"
!mkdir -p "$asv_cat_adjusted_pairwise_path" "$kmer16_cat_adjusted_pairwise_path"

### 1.1.3 Load inputs

We load the whole-ASV and 16-mer Bray–Curtis distance matrices from notebook 07 and the cleaned metadata from notebook 06.

In [20]:
metadata = q2.Metadata.load(metadata_path).to_dataframe()
kmer16_distance_matrix = q2.Artifact.load(kmer16_distance_path).view(DistanceMatrix)
asv_distance_matrix = q2.Artifact.load(asv_distance_path).view(DistanceMatrix)

## 1.2. Select samples

### 1.2.1 Match sample IDs

We keep samples present in both Bray–Curtis matrices and in the metadata.

In [21]:
columns = ["bmi_sample", "bmi_group", "age_years_sample", "diet_type_sample", "sex_sample"]

bmi_metadata = metadata[columns].dropna()

bmi_metadata.index.name = "sample-id"
q2.Metadata(bmi_metadata).save(f"{bmi_metadata_path}")

print("Complete samples:", len(bmi_metadata))

display(bmi_metadata["bmi_sample"].describe())
display(bmi_metadata["bmi_group"].value_counts())

Complete samples: 139


count    139.000000
mean      24.251727
std        6.097824
min       14.350000
25%       21.015000
50%       23.310000
75%       26.810000
max       71.610000
Name: bmi_sample, dtype: float64

bmi_group
Healthy weight    84
Overweight        33
Obese             12
Underweight       10
Name: count, dtype: int64

In [22]:
bmi_metadata.head()

,bmi_sample,bmi_group,age_years_sample,diet_type_sample,sex_sample
sample-id,,,,,
ERR5327198,17.75,Underweight,67.0,Omnivore,female
ERR5327199,23.73,Healthy weight,55.0,Omnivore,male
ERR5327266,19.94,Healthy weight,28.0,Omnivore,female
ERR5327282,21.77,Healthy weight,26.0,Omnivore,female
ERR5327284,20.01,Healthy weight,25.0,Vegetarian but eat seafood,female


### 1.2.2 Filter both Bray–Curtis matrices

We filter the whole-ASV and 16-mer matrices using the same complete-case metadata.

In [23]:
!qiime diversity filter-distance-matrix \
    --i-distance-matrix "$asv_distance_path" \
    --m-metadata-file "$bmi_metadata_path" \
    --o-filtered-distance-matrix "$asv_bmi_distance_path"

!qiime diversity filter-distance-matrix \
    --i-distance-matrix "$kmer16_distance_path" \
    --m-metadata-file "$bmi_metadata_path" \
    --o-filtered-distance-matrix "$kmer16_bmi_distance_path"

Saved DistanceMatrix to: ../data/processed_ignored/12_beta_diversity_bmi/adjusted_bmi/asv_bray_curtis_complete.qza
Saved DistanceMatrix to: ../data/processed_ignored/12_beta_diversity_bmi/adjusted_bmi/kmer16_bray_curtis_complete.qza


## 1.3. Overall PERMANOVA 

### 1.3.1 Unadjusted PERMANOVA

We test whether fungal community composition differs across BMI values. Both tests use the same 139 samples and include BMI as the only predictor.

#### 1.3.1.1 Whole ASVs

In [24]:
!qiime diversity adonis \
    --i-distance-matrix "$asv_bmi_distance_path" \
    --m-metadata-file "$bmi_metadata_path" \
    --p-formula "bmi_sample" \
    --o-visualization "$asv_con_unadjusted_overall_path"

Saved Visualization to: ../data/processed/12_beta_diversity_bmi/continuous_bmi/unadjusted_con_permanova/asv_overall.qzv


In [25]:
Visualization.load(asv_con_unadjusted_overall_path)

<visualization: Visualization uuid: 809d074d-8c8a-4ad0-96b4-b91b47e83b45>

#### 1.3.1.2 16-mers

In [26]:
!qiime diversity adonis \
    --i-distance-matrix "$adjusted_kmer_bmi_distance_path" \
    --m-metadata-file "$adjusted_bmi_metadata_path" \
    --p-formula "bmi_sample" \
    --o-visualization "$kmer16_con_unadjusted_overall_path"

Saved Visualization to: ../data/processed/12_beta_diversity_bmi/continuous_bmi/unadjusted_con_permanova/kmer16_overall.qzv


In [27]:
Visualization.load(kmer16_con_unadjusted_overall_path)

<visualization: Visualization uuid: 091a24ac-5eec-40fb-993c-ed6f593c272c>

#### Unadjusted results

| **BMI term** | **Whole ASVs** | **16-mers** |
| ------------ | -------------: | ----------: |
| Samples      |            139 |         139 |
| R²           |         0.0080 |      0.0072 |
| Pseudo-F     |          1.112 |       0.995 |
| p-value      |          0.320 |       0.455 |

Among the 139 samples included in both analyses, BMI explained less than 1% of the variation in fungal community composition (0.80% for whole ASVs and 0.72% for 16-mers). Both p-values were above 0.05.

### 1.3.2 Adjusted PERMANOVA

We put settlement type last in the formula to test whether fungal community composition correlates with BMI after accounting for age, diet and sex. Both tests use the same 139 samples as the unadjusted tests.

#### 1.3.2.1 Whole ASVs

In [28]:
!qiime diversity adonis \
    --i-distance-matrix "$asv_bmi_distance_path" \
    --m-metadata-file "$bmi_metadata_path" \
    --p-formula "age_years_sample + diet_type_sample + sex_sample + bmi_sample" \
    --o-visualization "$asv_con_adjusted_overall_path"

Saved Visualization to: ../data/processed/12_beta_diversity_bmi/continuous_bmi/adjusted_con_permanova/asv_overall.qzv


In [29]:
Visualization.load(asv_con_adjusted_overall_path)

<visualization: Visualization uuid: 229cf1af-21b3-438f-9648-068b14e829b6>

#### 1.3.2.2 16-mers

In [30]:
!qiime diversity adonis \
    --i-distance-matrix "$kmer16_bmi_distance_path" \
    --m-metadata-file "$bmi_metadata_path" \
    --p-formula "age_years_sample + diet_type_sample + sex_sample + bmi_sample" \
    --o-visualization "$kmer16_con_adjusted_overall_path"

Saved Visualization to: ../data/processed/12_beta_diversity_bmi/continuous_bmi/adjusted_con_permanova/kmer16_overall.qzv


In [31]:
Visualization.load(kmer16_con_adjusted_overall_path)

<visualization: Visualization uuid: f162060a-f070-4a56-82a7-1d9008b7903d>

#### Adjusted results

| **BMI term** | **Whole ASVs** | **16-mers** |
| ------------ | -------------: | ----------: |
| Samples      |            139 |         139 |
| R²           |         0.0078 |      0.0074 |
| Pseudo-F     |          1.079 |       1.017 |
| p-value      |          0.373 |       0.418 |

After accounting for age, sex, and diet, BMI explained less than 1% of the variation in fungal community composition (0.78% for whole ASVs and 0.74% for 16-mers). Both p-values were above 0.05.

### 1.3.3 Compare unadjusted and adjusted overall PERMANOVA

We compare the BMI term before and after accounting for age, sex, and diet.

| **Bray–Curtis representation** | **Model**  | **BMI R²** | **Pseudo-F** | **p-value** |
| ------------------------------ | ---------- | ---------: | -----------: | ----------: |
| Whole ASVs                     | Unadjusted |     0.0080 |        1.112 |       0.320 |
| Whole ASVs                     | Adjusted   |     0.0078 |        1.079 |       0.373 |
| 16-mers                        | Unadjusted |     0.0072 |        0.995 |       0.455 |
| 16-mers                        | Adjusted   |     0.0074 |        1.017 |       0.418 |

After adjustment, BMI R² decreased slightly for whole ASVs and increased slightly for 16-mers.


# 2. BMI as a categorical predictor (non-numeric values)

In Section 1., we tested for a linear association between BMI and fungal community composition. We now ask whether community composition differs among the underweight, normal-weight, and overweight groups.

A single numeric BMI term models a steady shift in average community composition as BMI increases. The shift from underweight to normal weight could go in a different direction than the shift from normal weight to overweight. If those shifts partly cancel, the linear test could miss a real association, even if all three groups differ. A non-significant linear result therefore leaves open the possibility of differences among BMI categories.

The adjusted metadata file (`Final_metadata_with_GHSL_exclude_other_sex_with_BMI_groups.tsv`) containing a categorical BMI column ["Underweight", "Healthy weight", "Overweight", "Obese"] has already been created in the notebook `metadata_distributions` and can be imported as the second metadata file of this script (`metadata_cat_path`). In this second part of the script, we use BMI as a categorical predictor in Bray–Curtis PERMANOVA, allowing each group to have a distinct average community composition. We use the same samples for the whole-ASV and 16-mer analyses and account for age, diet, and sex in the adjusted model.

### 2.1.1 Imports

In [32]:
import subprocess
from pathlib import Path

import pandas as pd
import qiime2 as q2
from skbio import DistanceMatrix
from qiime2 import Visualization, Artifact, Metadata

from itertools import combinations # to create metadata combinations for pairwise PERMANOVA

### 2.1.2 Notebook settings and paths

We define the input paths, the shared complete-case files, and the outputs for each feature representation and model.

In [ ]:
# 1. Shared inputs and output directories
metadata_cat_path = "../data/raw/Final_metadata_with_GHSL_exclude_other_sex_with_BMI_groups.tsv"
kmer16_distance_path = "../data/processed_ignored/07_beta_diversity/kmer16_bray_curtis_distance.qza"
asv_distance_path = "../data/processed_ignored/07_beta_diversity/asv_bray_curtis_distance.qza"

out_dir = "../data/processed_ignored/12_beta_diversity_bmi"
vis_dir = "../data/processed/12_beta_diversity_bmi"

columns_categorical_BMI = ["age_years_sample", "diet_type_sample", "sex_sample", "bmi_group"]


# 2. Complete-case inputs for both models
adjusted_cat_bmi_out_dir = f"{out_dir}/adjusted_cat_bmi"

adjusted_cat_asv_bmi_distance_path = f"{adjusted_cat_bmi_out_dir}/asv_bray_curtis_complete.qza"
adjusted_cat_kmer_bmi_distance_path = f"{adjusted_cat_bmi_out_dir}/kmer16_bray_curtis_complete.qza"

adjusted_cat_bmi_metadata_path = f"{adjusted_cat_bmi_out_dir}/metadata_cat_complete.tsv"


# 3. Overall PERMANOVA outputs
unadjusted_cat_permanova_dir = f"{vis_dir}/unadjusted_cat_permanova"
adjusted_cat_permanova_dir = f"{vis_dir}/adjusted_cat_permanova"

asv_cat_unadjusted_overall_path = f"{unadjusted_cat_permanova_dir}/asv_overall.qzv"
kmer16_cat_unadjusted_overall_path = f"{unadjusted_cat_permanova_dir}/kmer16_overall.qzv"

asv_cat_adjusted_overall_path = f"{adjusted_cat_permanova_dir}/asv_overall.qzv"
kmer16_cat_adjusted_overall_path = f"{adjusted_cat_permanova_dir}/kmer16_overall.qzv"


# 4. Pairwise PERMANOVA outputs
pairwise_input_dir = f"{adjusted_cat_bmi_out_dir}/pairwise_inputs"

asv_cat_unadjusted_pairwise_path = f"{unadjusted_cat_permanova_dir}/asv_pairwise"
kmer16_cat_unadjusted_pairwise_path = f"{unadjusted_cat_permanova_dir}/kmer16_pairwise"

asv_cat_adjusted_pairwise_path = f"{adjusted_cat_permanova_dir}/asv_pairwise"
kmer16_cat_adjusted_pairwise_path = f"{adjusted_cat_permanova_dir}/kmer16_pairwise"


# Create directories
!mkdir -p "$out_dir" "$vis_dir" "$adjusted_cat_bmi_out_dir"
!mkdir -p "$unadjusted_cat_permanova_dir" "$adjusted_cat_permanova_dir"
!mkdir -p "$pairwise_input_dir" "$asv_cat_unadjusted_pairwise_path" "$kmer16_cat_unadjusted_pairwise_path"
!mkdir -p "$asv_cat_adjusted_pairwise_path" "$kmer16_cat_adjusted_pairwise_path"

## 2.3 Overall PERMANOVA 

### 2.3.1 Unadjusted PERMANOVA

We test whether fungal community composition differs across BMI categories. Both tests use the same 139 samples and include BMI as the only predictor.

#### 2.3.1.1 Whole ASVs

In [33]:
!qiime diversity adonis \
    --i-distance-matrix "$asv_bmi_distance_path" \
    --m-metadata-file "$bmi_metadata_path" \
    --p-formula "bmi_group" \
    --o-visualization "$asv_cat_unadjusted_overall_path"

Saved Visualization to: ../data/processed/12_beta_diversity_bmi/categorical_bmi/unadjusted_cat_permanova/asv_overall.qzv


In [34]:
Visualization.load(asv_cat_unadjusted_overall_path)

<visualization: Visualization uuid: daf1552e-84d4-4c54-ae7b-35f625b88edb>

#### 2.3.1.2 16-mers

In [35]:
!qiime diversity adonis \
    --i-distance-matrix "$kmer16_bmi_distance_path" \
    --m-metadata-file "$bmi_metadata_path" \
    --p-formula "bmi_group" \
    --o-visualization "$kmer16_cat_unadjusted_overall_path"

Saved Visualization to: ../data/processed/12_beta_diversity_bmi/categorical_bmi/unadjusted_cat_permanova/kmer16_overall.qzv


In [36]:
Visualization.load(kmer16_cat_unadjusted_overall_path)

<visualization: Visualization uuid: 41659e9c-d048-4ba6-91db-62c6185109d2>

#### Unadjusted results

| **BMI group term** | **Whole ASVs** | **16-mers** |
| ------------------ | -------------: | ----------: |
| Samples            |            139 |         139 |
| R²                 |         0.0207 |      0.0202 |
| Pseudo-F           |          0.950 |       0.928 |
| p-value            |          0.552 |       0.531 |

Among the 139 samples, BMI group explained 2.07% of the variation in fungal community composition for whole ASVs and 2.02% for 16-mers. Both p-values were above 0.05.

### 2.3.2 Adjusted PERMANOVA

We put categorical BMI last in the formula to test whether fungal community composition significantly shifts across BMI labels after accounting for age, diet and sex. Both tests use the same 139 samples as the unadjusted tests.

#### 2.3.2.1 Whole ASVs

In [37]:
!qiime diversity adonis \
    --i-distance-matrix "$asv_bmi_distance_path" \
    --m-metadata-file "$bmi_metadata_path" \
    --p-formula "age_years_sample + diet_type_sample + sex_sample + bmi_group" \
    --o-visualization "$asv_cat_adjusted_overall_path"

Saved Visualization to: ../data/processed/12_beta_diversity_bmi/categorical_bmi/adjusted_cat_permanova/asv_overall.qzv


In [38]:
Visualization.load(asv_cat_adjusted_overall_path)

<visualization: Visualization uuid: 6c37d562-f5ce-4bfa-9ce3-797e7be027c9>

#### 2.3.2.2 16-mers

In [39]:
!qiime diversity adonis \
    --i-distance-matrix "$kmer16_bmi_distance_path" \
    --m-metadata-file "$bmi_metadata_path" \
    --p-formula "age_years_sample + diet_type_sample + sex_sample + bmi_group" \
    --o-visualization "$kmer16_cat_adjusted_overall_path"

Saved Visualization to: ../data/processed/12_beta_diversity_bmi/categorical_bmi/adjusted_cat_permanova/kmer16_overall.qzv


In [40]:
Visualization.load(kmer16_cat_adjusted_overall_path)

<visualization: Visualization uuid: bc322319-6888-4bee-bda3-78faab5bff24>

#### Adjusted results

| **BMI group term** | **Whole ASVs** | **16-mers** |
| ------------------ | -------------: | ----------: |
| Samples            |            139 |         139 |
| R²                 |         0.0210 |      0.0201 |
| Pseudo-F           |          0.963 |       0.918 |
| p-value            |          0.530 |       0.544 |

After accounting for age, sex, and diet, BMI group explained 2.10% of the variation in fungal community composition for whole ASVs and 2.01% for 16-mers. Both p-values were above 0.05.

### 2.3.3 Compare unadjusted and adjusted overall PERMANOVA

We compare the BMI group term before and after accounting for age, sex, and diet.

| **Bray–Curtis representation** | **Model**  | **BMI group R²** | **Pseudo-F** | **p-value** |
| ------------------------------ | ---------- | ---------------: | -----------: | ----------: |
| Whole ASVs                     | Unadjusted |           0.0207 |        0.950 |       0.552 |
| Whole ASVs                     | Adjusted   |           0.0210 |        0.963 |       0.530 |
| 16-mers                        | Unadjusted |           0.0202 |        0.928 |       0.531 |
| 16-mers                        | Adjusted   |           0.0201 |        0.918 |       0.544 |

After adjustment, BMI group R² increased slightly for whole ASVs and decreased slightly for 16-mers. Both representations provided little evidence of an overall difference in fungal community composition among the four BMI groups.

## 2.4. Pairwise PERMANOVA

We compare all six pairs of BMI groups on the whole-ASV and 16-mer Bray–Curtis matrices. For each pair, we use the same samples in the unadjusted and adjusted tests. We apply Holm correction across the ten comparisons separately for each representation and model.

### Prepare the pairwise inputs

We start from the 139 complete cases. For each diet pair, we save a matching metadata file and subset both distance matrices to those samples.

In [41]:
BMI_pairs = list(combinations(sorted(bmi_metadata["bmi_group"].unique()), 2))

for pair_number, (group_a, group_b) in enumerate(BMI_pairs, start=1):
    pair_cat_metadata = bmi_metadata.loc[bmi_metadata["bmi_group"].isin([group_a, group_b])]
    pair_cat_ids = pair_cat_metadata.index.tolist()

    Metadata(pair_cat_metadata).save(f"{pairwise_input_dir}/pair_{pair_number}_cat_metadata.tsv")

    asv_pair = asv_distance_matrix.filter(pair_cat_ids)
    kmer16_pair = kmer16_distance_matrix.filter(pair_cat_ids)

    Artifact.import_data("DistanceMatrix", asv_pair).save(
        f"{pairwise_input_dir}/pair_{pair_number}_cat_asv.qza"
    )
    Artifact.import_data("DistanceMatrix", kmer16_pair).save(
        f"{pairwise_input_dir}/pair_{pair_number}_cat_kmer16.qza"
    )

    print(f"{pair_number}. {group_a} vs. {group_b} (n = {len(pair_cat_ids)})")

1. Healthy weight vs. Obese (n = 96)
2. Healthy weight vs. Overweight (n = 117)
3. Healthy weight vs. Underweight (n = 94)
4. Obese vs. Overweight (n = 45)
5. Obese vs. Underweight (n = 22)
6. Overweight vs. Underweight (n = 43)


### 2.4.1 Unadjusted pairwise PERMANOVA

#### 2.4.1.1 Whole ASVs

We run one PERMANOVA per diet pair using the whole-ASV Bray–Curtis distances.

In [43]:
for pair_number, (group_a, group_b) in enumerate(BMI_pairs, start=1):
    pair_distance_path = f"{pairwise_input_dir}/pair_{pair_number}_cat_asv.qza"
    pair_metadata_path = f"{pairwise_input_dir}/pair_{pair_number}_cat_metadata.tsv"
    pair_qzv_path = f"{asv_cat_unadjusted_pairwise_path}/asv_cat_pair_{pair_number:02d}.qzv"

    print(f"{pair_number}. {group_a} vs. {group_b}")

    !qiime diversity adonis \
        --i-distance-matrix "$pair_distance_path" \
        --m-metadata-file "$pair_metadata_path" \
        --p-formula "bmi_group" \
        --o-visualization "$pair_qzv_path"

1. Healthy weight vs. Obese
Saved Visualization to: ../data/processed/12_beta_diversity_bmi/categorical_bmi/unadjusted_cat_permanova/asv_pairwise/asv_cat_pair_01.qzv
2. Healthy weight vs. Overweight
Saved Visualization to: ../data/processed/12_beta_diversity_bmi/categorical_bmi/unadjusted_cat_permanova/asv_pairwise/asv_cat_pair_02.qzv
3. Healthy weight vs. Underweight
Saved Visualization to: ../data/processed/12_beta_diversity_bmi/categorical_bmi/unadjusted_cat_permanova/asv_pairwise/asv_cat_pair_03.qzv
4. Obese vs. Overweight
Saved Visualization to: ../data/processed/12_beta_diversity_bmi/categorical_bmi/unadjusted_cat_permanova/asv_pairwise/asv_cat_pair_04.qzv
5. Obese vs. Underweight
Saved Visualization to: ../data/processed/12_beta_diversity_bmi/categorical_bmi/unadjusted_cat_permanova/asv_pairwise/asv_cat_pair_05.qzv
6. Overweight vs. Underweight
Saved Visualization to: ../data/processed/12_beta_diversity_bmi/categorical_bmi/unadjusted_cat_permanova/asv_pairwise/asv_cat_pair_06.qz

In [ ]:
# Visualization.load(f"{asv_cat_unadjusted_pairwise_path}/kmer16_pair_01.qzv")
# Visualization.load(f"{asv_cat_unadjusted_pairwise_path}/kmer16_pair_02.qzv")
# Visualization.load(f"{asv_cat_unadjusted_pairwise_path}/kmer16_pair_03.qzv")
# Visualization.load(f"{asv_cat_unadjusted_pairwise_path}/kmer16_pair_04.qzv")
# Visualization.load(f"{asv_cat_unadjusted_pairwise_path}/kmer16_pair_05.qzv")
# Visualization.load(f"{asv_cat_unadjusted_pairwise_path}/kmer16_pair_06.qzv")

#### Unadjusted pairwise results · whole ASVs

| BMI pair | n | R² | Pseudo-F | Raw p | Holm-adjusted p |
| --- | ---: | ---: | ---: | ---: | ---: |
| Healthy weight vs. Obese | 96 | 0.0094 | 0.895 | 0.535 | 1.000 |
| Healthy weight vs. Overweight | 117 | 0.0076 | 0.882 | 0.544 | 1.000 |
| Healthy weight vs. Underweight | 94 | 0.0108 | 1.007 | 0.420 | 1.000 |
| Obese vs. Overweight | 45 | 0.0200 | 0.876 | 0.538 | 1.000 |
| Obese vs. Underweight | 22 | 0.0591 | 1.257 | 0.210 | 1.000 |
| Overweight vs. Underweight | 43 | 0.0254 | 1.068 | 0.344 | 1.000 |

None of the six exploratory pairwise comparisons was statistically significant after Holm correction. The obese vs. underweight comparison had the largest observed R² (5.91%) and the smallest raw p-value (0.210).

#### 2.4.1.2 16-mers

We repeat the ten unadjusted pairwise PERMANOVAs using 16-mer Bray–Curtis distances and the same samples for each diet pair. Diet is the only predictor.

In [44]:
for pair_number, (group_a, group_b) in enumerate(BMI_pairs, start=1):
    pair_distance_path = f"{pairwise_input_dir}/pair_{pair_number}_cat_kmer16.qza"
    pair_metadata_path = f"{pairwise_input_dir}/pair_{pair_number}_cat_metadata.tsv"
    pair_qzv_path = f"{kmer16_cat_unadjusted_pairwise_path}/kmer16_cat_pair_{pair_number:02d}.qzv"

    print(f"{pair_number}. {group_a} vs. {group_b}")

    !qiime diversity adonis \
        --i-distance-matrix "$pair_distance_path" \
        --m-metadata-file "$pair_metadata_path" \
        --p-formula "bmi_group" \
        --o-visualization "$pair_qzv_path"

1. Healthy weight vs. Obese
Saved Visualization to: ../data/processed/12_beta_diversity_bmi/categorical_bmi/unadjusted_cat_permanova/kmer16_pairwise/kmer16_cat_pair_01.qzv
2. Healthy weight vs. Overweight
Saved Visualization to: ../data/processed/12_beta_diversity_bmi/categorical_bmi/unadjusted_cat_permanova/kmer16_pairwise/kmer16_cat_pair_02.qzv
3. Healthy weight vs. Underweight
Saved Visualization to: ../data/processed/12_beta_diversity_bmi/categorical_bmi/unadjusted_cat_permanova/kmer16_pairwise/kmer16_cat_pair_03.qzv
4. Obese vs. Overweight
Saved Visualization to: ../data/processed/12_beta_diversity_bmi/categorical_bmi/unadjusted_cat_permanova/kmer16_pairwise/kmer16_cat_pair_04.qzv
5. Obese vs. Underweight
Saved Visualization to: ../data/processed/12_beta_diversity_bmi/categorical_bmi/unadjusted_cat_permanova/kmer16_pairwise/kmer16_cat_pair_05.qzv
6. Overweight vs. Underweight
Saved Visualization to: ../data/processed/12_beta_diversity_bmi/categorical_bmi/unadjusted_cat_permanova/k

In [ ]:
# Visualization.load(f"{kmer16_cat_unadjusted_pairwise_path}/kmer16_cat_pair_01.qzv")
# Visualization.load(f"{kmer16_cat_unadjusted_pairwise_path}/kmer16_cat_pair_02.qzv")
# Visualization.load(f"{kmer16_cat_unadjusted_pairwise_path}/kmer16_cat_pair_03.qzv")
# Visualization.load(f"{kmer16_cat_unadjusted_pairwise_path}/kmer16_cat_pair_04.qzv")
# Visualization.load(f"{kmer16_cat_unadjusted_pairwise_path}/kmer16_cat_pair_05.qzv")
# Visualization.load(f"{kmer16_cat_unadjusted_pairwise_path}/kmer16_cat_pair_06.qzv")

#### Unadjusted pairwise results · 16-mers

| BMI pair | n | R² | Pseudo-F | Raw p | Holm-adjusted p |
| --- | ---: | ---: | ---: | ---: | ---: |
| Healthy weight vs. Obese | 96 | 0.0111 | 1.053 | 0.324 | 1.000 |
| Healthy weight vs. Overweight | 117 | 0.0067 | 0.774 | 0.610 | 1.000 |
| Healthy weight vs. Underweight | 94 | 0.0094 | 0.872 | 0.500 | 1.000 |
| Obese vs. Overweight | 45 | 0.0249 | 1.100 | 0.299 | 1.000 |
| Obese vs. Underweight | 22 | 0.0494 | 1.040 | 0.349 | 1.000 |
| Overweight vs. Underweight | 43 | 0.0231 | 0.968 | 0.427 | 1.000 |

None of the six exploratory pairwise comparisons was statistically significant after Holm correction. The obese vs. underweight comparison had the largest observed R² (4.94%), and the obese vs. overweight comparison had the smallest raw p-value (0.299).

#### Unadjusted results

| BMI pair | Whole-ASV R² | 16-mer R² | Whole-ASV Holm-adjusted p | 16-mer Holm-adjusted p |
| --- | ---: | ---: | ---: | ---: |
| Obese vs. Underweight | 0.0591 | 0.0494 | 1.000 | 1.000 |
| Healthy weight vs. Underweight | 0.0108 | 0.0094 | 1.000 | 1.000 |

Obese vs. Underweight had the highest observed R² across all six pairwise comparisons in both representations (5.91% for whole ASVs and 4.94% for 16-mers). Healthy weight vs. Underweight had the lowest R² in both representations (1.08% and 0.94%, respectively). All six comparisons in each representation had a Holm-adjusted p-value of 1.000.

### 2.4.2 Adjusted pairwise PERMANOVA

We reuse the six BMI pairs and sample subsets from section 2.4.1. For each pair, we test the BMI term after accounting for age, diet and sex. We apply Holm correction across the six comparisons separately for whole ASVs and 16-mers.

#### 2.4.2.1 Whole ASVs

In [45]:
for pair_number, (group_a, group_b) in enumerate(BMI_pairs, start=1):
    pair_distance_path = f"{pairwise_input_dir}/pair_{pair_number}_cat_asv.qza"
    pair_metadata_path = f"{pairwise_input_dir}/pair_{pair_number}_cat_metadata.tsv"
    pair_qzv_path = f"{asv_cat_adjusted_pairwise_path}/asv_cat_pair_{pair_number:02d}.qzv"

    print(f"{pair_number}. {group_a} vs. {group_b}")

    !qiime diversity adonis \
        --i-distance-matrix "$pair_distance_path" \
        --m-metadata-file "$pair_metadata_path" \
        --p-formula "age_years_sample + diet_type_sample + sex_sample + bmi_group" \
        --o-visualization "$pair_qzv_path"

1. Healthy weight vs. Obese
Saved Visualization to: ../data/processed/12_beta_diversity_bmi/categorical_bmi/adjusted_cat_permanova/asv_pairwise/asv_cat_pair_01.qzv
2. Healthy weight vs. Overweight
Saved Visualization to: ../data/processed/12_beta_diversity_bmi/categorical_bmi/adjusted_cat_permanova/asv_pairwise/asv_cat_pair_02.qzv
3. Healthy weight vs. Underweight
Saved Visualization to: ../data/processed/12_beta_diversity_bmi/categorical_bmi/adjusted_cat_permanova/asv_pairwise/asv_cat_pair_03.qzv
4. Obese vs. Overweight
Saved Visualization to: ../data/processed/12_beta_diversity_bmi/categorical_bmi/adjusted_cat_permanova/asv_pairwise/asv_cat_pair_04.qzv
5. Obese vs. Underweight
Saved Visualization to: ../data/processed/12_beta_diversity_bmi/categorical_bmi/adjusted_cat_permanova/asv_pairwise/asv_cat_pair_05.qzv
6. Overweight vs. Underweight
Saved Visualization to: ../data/processed/12_beta_diversity_bmi/categorical_bmi/adjusted_cat_permanova/asv_pairwise/asv_cat_pair_06.qzv


In [ ]:
# Visualization.load(f"{asv_cat_adjusted_pairwise_path}/asv_cat_pair_01.qzv")
# Visualization.load(f"{asv_cat_adjusted_pairwise_path}/asv_cat_pair_02.qzv")
# Visualization.load(f"{asv_cat_adjusted_pairwise_path}/asv_cat_pair_03.qzv")
# Visualization.load(f"{asv_cat_adjusted_pairwise_path}/asv_cat_pair_04.qzv")
# Visualization.load(f"{asv_cat_adjusted_pairwise_path}/asv_cat_pair_05.qzv")
# Visualization.load(f"{asv_cat_adjusted_pairwise_path}/asv_cat_pair_06.qzv")

#### Adjusted pairwise results · Whole ASVs

| BMI pair | n | R² | Pseudo-F | Raw p | Holm-adjusted p |
| --- | ---: | ---: | ---: | ---: | ---: |
| Healthy weight vs. Obese | 96 | 0.0105 | 0.993 | 0.420 | 1.000 |
| Healthy weight vs. Overweight | 117 | 0.0084 | 0.973 | 0.436 | 1.000 |
| Healthy weight vs. Underweight | 94 | 0.0096 | 0.891 | 0.519 | 1.000 |
| Obese vs. Overweight | 45 | 0.0210 | 0.920 | 0.495 | 1.000 |
| Obese vs. Underweight | 22 | 0.0663 | 1.436 | 0.147 | 0.882 |
| Overweight vs. Underweight | 43 | 0.0246 | 1.047 | 0.370 | 1.000 |

After accounting for age, sex, and diet, none of the six exploratory pairwise comparisons was statistically significant after Holm correction. Obese vs. Underweight had the highest observed R² (6.63%) and the lowest raw p-value (0.147, Holm-adjusted p = 0.882).

#### 2.4.2.2 16-mers

We repeat the ten adjusted pairwise PERMANOVAs using 16-mer Bray–Curtis distances and the same sample subsets as in 2.4.2.1. We test the diet term after accounting for age, diet and sex.

In [46]:
for pair_number, (group_a, group_b) in enumerate(BMI_pairs, start=1):
    pair_distance_path = f"{pairwise_input_dir}/pair_{pair_number}_cat_kmer16.qza"
    pair_metadata_path = f"{pairwise_input_dir}/pair_{pair_number}_cat_metadata.tsv"
    pair_qzv_path = f"{kmer16_cat_adjusted_pairwise_path}/kmer16_cat_pair_{pair_number:02d}.qzv"

    print(f"{pair_number}. {group_a} vs. {group_b}")

    !qiime diversity adonis \
        --i-distance-matrix "$pair_distance_path" \
        --m-metadata-file "$pair_metadata_path" \
        --p-formula "age_years_sample + diet_type_sample + sex_sample + bmi_group" \
        --o-visualization "$pair_qzv_path"

1. Healthy weight vs. Obese
Saved Visualization to: ../data/processed/12_beta_diversity_bmi/categorical_bmi/adjusted_cat_permanova/kmer16_pairwise/kmer16_cat_pair_01.qzv
2. Healthy weight vs. Overweight
Saved Visualization to: ../data/processed/12_beta_diversity_bmi/categorical_bmi/adjusted_cat_permanova/kmer16_pairwise/kmer16_cat_pair_02.qzv
3. Healthy weight vs. Underweight
Saved Visualization to: ../data/processed/12_beta_diversity_bmi/categorical_bmi/adjusted_cat_permanova/kmer16_pairwise/kmer16_cat_pair_03.qzv
4. Obese vs. Overweight
Saved Visualization to: ../data/processed/12_beta_diversity_bmi/categorical_bmi/adjusted_cat_permanova/kmer16_pairwise/kmer16_cat_pair_04.qzv
5. Obese vs. Underweight
Saved Visualization to: ../data/processed/12_beta_diversity_bmi/categorical_bmi/adjusted_cat_permanova/kmer16_pairwise/kmer16_cat_pair_05.qzv
6. Overweight vs. Underweight
Saved Visualization to: ../data/processed/12_beta_diversity_bmi/categorical_bmi/adjusted_cat_permanova/kmer16_pairwi

In [ ]:
# Visualization.load(f"{kmer16_cat_adjusted_pairwise_path}/kmer16_cat_pair_01.qzv")
# Visualization.load(f"{kmer16_cat_adjusted_pairwise_path}/kmer16_cat_pair_02.qzv")
# Visualization.load(f"{kmer16_cat_adjusted_pairwise_path}/kmer16_cat_pair_03.qzv")
# Visualization.load(f"{kmer16_cat_adjusted_pairwise_path}/kmer16_cat_pair_04.qzv")
# Visualization.load(f"{kmer16_cat_adjusted_pairwise_path}/kmer16_cat_pair_05.qzv")
# Visualization.load(f"{kmer16_cat_adjusted_pairwise_path}/kmer16_cat_pair_06.qzv")

#### Adjusted pairwise results · 16-mers

| BMI pair | n | R² | Pseudo-F | Raw p | Holm-adjusted p |
| --- | ---: | ---: | ---: | ---: | ---: |
| Healthy weight vs. Obese | 96 | 0.0140 | 1.327 | 0.218 | 1.000 |
| Healthy weight vs. Overweight | 117 | 0.0074 | 0.853 | 0.512 | 1.000 |
| Healthy weight vs. Underweight | 94 | 0.0071 | 0.655 | 0.754 | 1.000 |
| Obese vs. Overweight | 45 | 0.0238 | 1.041 | 0.345 | 1.000 |
| Obese vs. Underweight | 22 | 0.0426 | 0.937 | 0.438 | 1.000 |
| Overweight vs. Underweight | 43 | 0.0222 | 0.929 | 0.469 | 1.000 |

After accounting for age, sex, and diet, none of the six exploratory pairwise comparisons was statistically significant after Holm correction. Obese vs. Underweight had the highest observed R² (4.26%), and Healthy weight vs. Obese had the lowest raw p-value (0.218). All Holm-adjusted p-values were 1.000.

#### Adjusted results

| BMI pair | Whole-ASV R² | 16-mer R² | Whole-ASV Holm-adjusted p | 16-mer Holm-adjusted p |
| --- | ---: | ---: | ---: | ---: |
| Obese vs. Underweight | 0.0663 | 0.0426 | 0.882 | 1.000 |
| Healthy weight vs. Obese | 0.0105 | 0.0140 | 1.000 | 1.000 |

After accounting for age, sex, and diet, Obese vs. Underweight had the highest observed R² in both representations and the largest gap between them (2.37 percentage points). For Healthy weight vs. Obese, R² was higher for 16-mers than for whole ASVs, and this pair had the lowest raw p-value among the 16-mer tests (0.218). None of the six adjusted pairwise comparisons was statistically significant after Holm correction in either representation.

### 2.4.3 Compare unadjusted and adjusted pairwise PERMANOVA

Both models used the same samples for each BMI pair. Holm-adjusted p-values ranged from 0.882 to 1.000 across the six comparisons in both representations. The R² values and raw p-values below are descriptive.

| BMI pair | Representation | BMI group R² (unadjusted → adjusted) | Raw p (unadjusted → adjusted) |
|---|---|---:|---:|
| Obese vs. Underweight (n = 22) | Whole ASVs | 0.0591 → 0.0663 | 0.210 → 0.147 |
| Obese vs. Underweight (n = 22) | 16-mers | 0.0494 → 0.0426 | 0.349 → 0.438 |
| Healthy weight vs. Obese (n = 96) | Whole ASVs | 0.0094 → 0.0105 | 0.535 → 0.420 |
| Healthy weight vs. Obese (n = 96) | 16-mers | 0.0111 → 0.0140 | 0.324 → 0.218 |

Obese vs. Underweight had the highest observed R² in both representations in both models. After accounting for age, sex, and diet, its R² increased for whole ASVs and decreased for 16-mers. Healthy weight vs. Obese had the lowest raw p-value among the adjusted 16-mer tests. Across all six BMI pairs, whole-ASV R² was higher than 16-mer R² in four comparisons in each model. None of the pairwise comparisons was statistically significant after Holm correction.

## 3. Discussion

The Bray–Curtis PERMANOVAs provided little evidence of an association between BMI and fungal community composition, whether BMI was treated as a continuous predictor or grouped into four categories. Continuous BMI explained less than 1% of the variation, and BMI groups explained about 2%. Adjusting for age, sex, and diet changed these estimates only slightly in both the whole-ASV and 16-mer analyses.

Obese vs. Underweight had the highest observed pairwise R² in both representations, before and after covariate adjustment. This comparison also had the smallest sample size (n = 22). None of the six pairwise comparisons reached statistical significance after Holm correction, so the higher R² for this pair is a purely descriptive finding.